In [ ]:
#STEP 1 - DATA LOADING & INSPECTION
# Step 1.a — Data Loading (Yahoo Finance)
# ----------------------------------------
# Goal: Pull raw daily market data for our variables:
# - BTC (crypto proxy)
# - NASDAQ (risk-on proxy)
# - DXY (USD strength proxy)
#
# Why we set START/END: reproducibility + consistent sample window

START = "2017-01-01"
END   = "2025-12-31"

# Yahoo ticker symbols (labels help readability later)
TICKERS = {
    "btc": "BTC-USD",     # Bitcoin price in USD
    "nasdaq": "^IXIC",    # NASDAQ Composite Index
    "dxy": "DX-Y.NYB"     # US Dollar Index (DXY)
}

import yfinance as yf
import pandas as pd

# Download daily OHLCV data:
# Open, High, Low, Close, Adj Close, Volume
# We keep raw output as *_raw so we don't overwrite cleaned data later
btc_raw = yf.download(TICKERS["btc"], start=START, end=END, auto_adjust=False, progress=False)
nasdaq_raw = yf.download(TICKERS["nasdaq"], start=START, end=END, auto_adjust=False, progress=False)
dxy_raw = yf.download(TICKERS["dxy"], start=START, end=END, auto_adjust=False, progress=False)

# Shape = (rows, columns)
# Rows = number of days available
# Columns ≈ 6 (OHLCV)
btc_raw.shape, nasdaq_raw.shape, dxy_raw.shape

In [ ]:
# Step 1.b — Data Loading (FRED macro data)
# ----------------------------------------
# Fed Funds is a macroeconomic policy rate (monthly series in FRED).
# We use FRED because it is an official macro source and cleaner than Yahoo for rates.
# Later we will resample monthly → daily to align with BTC/NASDAQ/DXY.

from pandas_datareader import data as pdr

fed_raw = pdr.DataReader("FEDFUNDS", "fred", START, END)

# Shape here should be approx. (months, 1)
fed_raw.shape

In [ ]:
# Step 1.c — Inspection & Wrangling Preparation
# ----------------------------------------
# Why reset_index():
# - yfinance returns Date as an index
# - we need Date as a column for merging (pd.merge requires a key column)
#
# Why flatten columns:
# - yfinance can return MultiIndex columns (e.g., ('Adj Close', 'BTC-USD'))
# - flattening ensures column names like "Adj Close" exist as plain text
# ----------------------------------------

def reset_and_flatten(df):
    df = df.reset_index()  # Date becomes a regular column
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level (e.g., "Adj Close")
    return df

btc_raw = reset_and_flatten(btc_raw)
nasdaq_raw = reset_and_flatten(nasdaq_raw)
dxy_raw = reset_and_flatten(dxy_raw)

# Fed data already has simple columns, just reset index
fed_raw = fed_raw.reset_index()

# Quick inspection of column names (helps debugging and confirms structure)
btc_raw.columns, nasdaq_raw.columns, dxy_raw.columns, fed_raw.columns

In [ ]:
# Step 1.d — Column Selection (Reduce dataset)
# ----------------------------------------
# We only need:
# - Date (merge key)
# - Adjusted Close (price level used for return calculations)
#
# Using Adj Close is standard because it accounts for corporate actions in equities.
# For Bitcoin, Close ≈ Adj Close, but we keep Adj Close for consistency.

btc = btc_raw[["Date", "Adj Close"]].rename(columns={"Adj Close": "btc_price"})
nasdaq = nasdaq_raw[["Date", "Adj Close"]].rename(columns={"Adj Close": "nasdaq_price"})
dxy = dxy_raw[["Date", "Adj Close"]].rename(columns={"Adj Close": "dxy_price"})

# Rename Fed columns for clarity
# FRED returns something like ["DATE", "FEDFUNDS"] after reset_index
fed_raw.columns = ["Date", "fed_rate"]
fed = fed_raw[["Date", "fed_rate"]]

# Quick preview (inspection step from lecture slides)
btc.head(), nasdaq.head(), dxy.head(), fed.head()

In [130]:
# Step 1.e — Data Type Cleaning (Datetime)
# ----------------------------------------
# Dates must match in type and format for:
# - merging
# - resampling
# - rolling window calculations

btc["Date"] = pd.to_datetime(btc["Date"])
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"])
dxy["Date"] = pd.to_datetime(dxy["Date"])
fed["Date"] = pd.to_datetime(fed["Date"])

In [ ]:
# Step 1.f — Data Quality Checks (Inspection)
# ----------------------------------------
# Before merging, we check:
# - missing values (NA)
# - duplicate dates (merge keys must be unique)
# These checks are important to avoid incorrect merges and biased results.

def quality_check(df, name):
    print(f"\n{name}")
    print("Missing values:\n", df.isna().sum())
    print("Duplicate dates:", df["Date"].duplicated().sum())
    print("Date range:", df["Date"].min(), "to", df["Date"].max())

quality_check(btc, "BTC")
quality_check(nasdaq, "NASDAQ")
quality_check(dxy, "DXY")
quality_check(fed, "FED FUNDS")

In [ ]:
#STEP 2 — Data Cleaning & Alignment
# ----------------------------------------
# Step 2.a — Sort by Date + verify unique merge keys
# ----------------------------------------
# Why:
# - Sorting is good practice for time-series work (resample/rolling later)
# - Unique Date keys are required for a valid 1:1 merge

btc = btc.sort_values("Date").reset_index(drop=True)
nasdaq = nasdaq.sort_values("Date").reset_index(drop=True)
dxy = dxy.sort_values("Date").reset_index(drop=True)
fed = fed.sort_values("Date").reset_index(drop=True)

# Check duplicates (should be 0; if not, we must fix before merge)
btc["Date"].duplicated().sum(), nasdaq["Date"].duplicated().sum(), dxy["Date"].duplicated().sum(), fed["Date"].duplicated().sum()

In [ ]:
# Step 2.b — Resample Fed Funds (monthly → daily)
# Why:
# - BTC/NASDAQ/DXY are daily series
# - FEDFUNDS is monthly
# - Resampling to daily + forward-fill means:
#   each day uses the most recently available policy rate until the next update

fed_daily = (
    fed.set_index("Date")
       .resample("D")      # create daily date index
       .ffill()            # forward-fill monthly values across days
       .reset_index()
)
fed_daily.head(), fed_daily.tail()

In [134]:
# Step 2.c — Align calendars (common trading days)
# ----------------------------------------
# Why:
# - BTC trades every day (including weekends)
# - NASDAQ/DXY trade mostly weekdays
# - For regression, we want dates where all variables exist
# Inner join = keep only dates where all variables exist.
# - Hence, INNER merges will be used (common dates only)

# Now it will be merged step-by-step using Date as the key

In [ ]:
# Step 2.d — Merge BTC and NASDAQ (1:1 validation + merge indicator)
# ----------------------------------------
# validate='1:1' ensures each Date appears at most once in each dataset
# indicator=True creates a _merge column showing how rows matched

df = pd.merge(
    btc, nasdaq,
    on="Date",
    how="inner",
    validate="1:1",
    indicator=True
)

df["_merge"].value_counts(), df.head()

In [ ]:
# Step 2.e — Merge DXY into the dataset
# Like we merged BTC and NASDAQ, we merge DXY using an inner join to keep only common dates.

df = pd.merge(
    df.drop(columns=["_merge"]),  # remove old indicator to avoid confusion
    dxy,
    on="Date",
    how="inner",
    validate="1:1",
    indicator=True
)

df["_merge"].value_counts(), df.head()

In [ ]:
# Step 2.f — Now Merge Fed Funds (daily resampled) into the dataset

df = pd.merge(
    df.drop(columns=["_merge"]),
    fed_daily,
    on="Date",
    how="inner",
    validate="1:1",
    indicator=True
)

df["_merge"].value_counts(), df.head(), df.tail()

In [ ]:
# Step 2.g — Post-merge quality check
# check for missing values and final shape of the dataset

df.isna().sum(), df.shape

In [ ]:
# Step 2.h — Export aligned dataset (processed)
# Useful for transparency + teamwork and for future steps (e.g., return calculations, regression)

df = df.drop(columns=["_merge"], errors="ignore")  # remove merge flag before saving and ignore if it doesn't exist
# We drop the _merge column before saving since it's only for debugging/inspection during merging.
# Create directory if it doesn't exist (it avoids errors when saving)
import os

os.makedirs("data/processed", exist_ok=True)

df.to_csv("data/processed/aligned_prices.csv", index=False)
df.to_csv("data/processed/aligned_prices.csv", index=False)

df.head(), df.tail()

In [140]:
# Step 2.I, CLARIFICATION, Actually, previous to the output from the step 2.h, the fed_rate was returning twice the colums--
# --indicating it was merged twice before, so in the following cell, the code--
# df["fed_rate"] = df["fed_rate_x"] was run to check if the different/duplicate columns were identical,-- 
# -- and if they were (which they were), the duplicate column was dropped with 
# df.drop(columns=["fed_rate_x", "fed_rate_y"], inplace=True
# Now this cannot be seen in the code above because it was fixed before the final output,
# but it is worth noting that this issue was encountered and resolved during the data cleaning process.

In [ ]:
# STEP 3 — Feature Engineering
# ----------------------------------------
# Step 3.a — Compute daily log returns
# Why:
# - Returns remove trend from price levels
# - Log returns are standard in financial econometrics
# - We use diff(log(price)) ≈ percentage change

import numpy as np

df["btc_ret"] = np.log(df["btc_price"]).diff()
df["nasdaq_ret"] = np.log(df["nasdaq_price"]).diff()
df["dxy_ret"] = np.log(df["dxy_price"]).diff()

df[["Date", "btc_ret", "nasdaq_ret", "dxy_ret"]].head()

In [ ]:
# Step 3.b — Compute daily change in Fed Funds rate
# Why:
# - Markets react to rate changes, not static levels 
# - First difference makes the series dynamic

df["fed_change"] = df["fed_rate"].diff()
(
    df[["Date", "fed_rate", "fed_change"]].head(10),
    df[["Date", "fed_rate", "fed_change"]].tail(10)
)

In [ ]:
# Step 3.c — Remove NA values from differencing.
# In previous outputs (3.a and 3.b) it can be seen that returns + fed_change introduced NA in first row,
# so we drop NA values to have a clean dataset for regression and analysis.

df = df.dropna().reset_index(drop=True)

df.shape

In [ ]:
# Step 3.d — 30-day rolling correlation (BTC vs NASDAQ)
# # Measures time-varying relationship between Bitcoin and equity market (NASDAQ)
# Shows how relationship evolves over time

df["rolling_corr_btc_nasdaq"] = (
    df["btc_ret"]
      .rolling(window=30)
      .corr(df["nasdaq_ret"])
)

df[["Date", "rolling_corr_btc_nasdaq"]].tail()

In [ ]:
# Step 3.E – Rolling 30-day correlation (BTC vs DXY)
# Tests whether Bitcoin behaves inversely to dollar strength

df["corr_btc_dxy"] = (
    df["btc_ret"]
    .rolling(window=30)
    .corr(df["dxy_ret"])
)

df[["Date", "corr_btc_dxy"]].tail()

In [ ]:
# Step 3.f — Summary statistics of engineered features
df[["btc_ret","nasdaq_ret","dxy_ret","fed_change"]].describe()

In [ ]:
# Step 4 - Statistical Analysis & Visualization

# Step 4.a – Correlation Matrix (Preliminary Relationship Assessment)

# This step evaluates the linear association between Bitcoin returns
# and the selected macro-financial variables.
# It provides initial descriptive evidence before formal regression.
# Note: Correlation does NOT imply causation.

corr_matrix = df[["btc_ret", "nasdaq_ret", "dxy_ret", "fed_change"]].corr()

corr_matrix

In [ ]:
# Step 4.b – OLS Regression Model

# Objective:
# Estimate the sensitivity (beta coefficients) of Bitcoin returns
# to macro-financial factors:
#   - NASDAQ returns (risk sentiment proxy)
#   - DXY returns (US dollar strength proxy)
#   - Fed Funds changes (monetary policy shock proxy)
#
# Model specification:
#   BTC_t = α + β1*NASDAQ_t + β2*DXY_t + β3*FedChange_t + ε_t
#
# OLS estimates the average marginal effect of each variable
# while holding the others constant.

import statsmodels.api as sm

# Define independent variables (macro factors)
X = df[["nasdaq_ret", "dxy_ret", "fed_change"]]

# Add constant term (intercept α)
X = sm.add_constant(X)

# Define dependent variable (Bitcoin returns)
y = df["btc_ret"]

# Fit Ordinary Least Squares model
model = sm.OLS(y, X).fit()

# Display regression summary
model.summary()

In [ ]:
# Step 4.c.1 – 30-Day Rolling Correlation (BTC vs NASDAQ) - Taken from step 3.d but now plotted for visualization.
# ---------------------------------------------------------------
# This measures the time-varying relationship between Bitcoin
# and equity market risk sentiment.
# A rolling window allows us to observe whether correlation
# strengthens during crises or weakens during decoupling periods.

import matplotlib.pyplot as plt

plt.figure()
plt.plot(df["Date"], df["rolling_corr_btc_nasdaq"])
plt.axhline(0)
plt.title("30-Day Rolling Correlation: BTC vs NASDAQ")
plt.xlabel("Date")
plt.ylabel("Correlation")
plt.show()

In [ ]:
# Step 4.c.2 – 30-Day Rolling Correlation (BTC vs DXY) - Taken from step 3.e but now plotted for visualization.
# ---------------------------------------------------------------
# This examines whether Bitcoin behaves inversely to the US dollar.
# Negative correlation suggests Bitcoin may act as a dollar hedge,
# while positive correlation suggests risk-on co-movement.

plt.figure()
plt.plot(df["Date"], df["corr_btc_dxy"])
plt.axhline(0)
plt.title("30-Day Rolling Correlation: BTC vs DXY")
plt.xlabel("Date")
plt.ylabel("Correlation")
plt.show()